# TrafficEye AI — YOLO26 training on Kaggle
Trains the 8-class model (Bike, Car, Jeep, Van, Helmet, No-Helmet, Seatbelt, No-Seatbelt) on a Kaggle GPU, evaluates it and exports the weights.

**Settings → Accelerator → GPU (T4 x2 or P100)**, and attach your dataset (YOLO format) as a Kaggle input.

In [ ]:
!pip install -q -U ultralytics
import ultralytics, torch
ultralytics.checks()
print('GPUs:', torch.cuda.device_count())

In [ ]:
import yaml, os
DATA_ROOT = '/kaggle/input/trafficeye'   # <- change to your dataset input
names = ['Bike','Car','Jeep','Van','Helmet','No-Helmet','Seatbelt','No-Seatbelt']
cfg = {'path': DATA_ROOT, 'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
       'nc': len(names), 'names': dict(enumerate(names))}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
for split in ('train', 'val', 'test'):
    d = os.path.join(DATA_ROOT, 'images', split)
    print(split, len(os.listdir(d)) if os.path.isdir(d) else 'missing')

## Class balance check
Violation classes are usually rarer than compliant ones; check before training.

In [ ]:
from collections import Counter
from pathlib import Path
cnt = Counter()
for f in Path(DATA_ROOT, 'labels', 'train').glob('*.txt'):
    for line in f.read_text().splitlines():
        if line.strip():
            cnt[names[int(line.split()[0])]] += 1
for n in names:
    print(f'{n:<12} {cnt[n]}')

## Train

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26s.pt')
model.train(data='/kaggle/working/data.yaml', epochs=120, imgsz=960, batch=16, device='0,1' if torch.cuda.device_count() > 1 else 0,
            project='/kaggle/working/runs', name='trafficeye_yolo26', patience=30, cos_lr=True, close_mosaic=10,
            hsv_h=0.015, hsv_s=0.6, hsv_v=0.45, degrees=3.0, translate=0.1, scale=0.5, shear=1.0,
            perspective=0.0005, fliplr=0.5, mosaic=1.0, mixup=0.1, copy_paste=0.1, seed=42)

## Evaluate (use these numbers in your write-up)

In [ ]:
best = YOLO('/kaggle/working/runs/trafficeye_yolo26/weights/best.pt')
m = best.val(data='/kaggle/working/data.yaml', split='test', imgsz=960)
print(f'mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f} | P {m.box.mp:.3f} | R {m.box.mr:.3f}')
print('speed (ms/img):', m.speed)

## Export

In [ ]:
import shutil
shutil.copy('/kaggle/working/runs/trafficeye_yolo26/weights/best.pt', '/kaggle/working/trafficeye_yolo26.pt')
best.export(format='onnx', imgsz=960, dynamic=True)
print('Download trafficeye_yolo26.pt from the Output tab and put it in weights/')